In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
# 1.Imports
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
import os
from dotenv import load_dotenv

# 2.Load API keys
load_dotenv(".env")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


In [4]:
# 3.Setup LLM

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)


In [5]:
# Create Vector DB (Retriever)
with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()


In [6]:

# 🧠 Split the text into smaller chunks
splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
texts = splitter.split_text(text_data)
embedding = OpenAIEmbeddings()
vectorstore = FAISS.from_texts(texts, embedding)
retriever = vectorstore.as_retriever()

In [8]:

# 5.Wrap the retriever as an agent tool (replaces RetrievalQA + Tool)
@tool
def LangChainRetriever(query: str) -> str:
    """Use this to answer questions about LangChain framework, features, or its creator."""
    docs = retriever.invoke(query)
    context = "\n\n".join(doc.page_content for doc in docs)
    answer = llm.invoke(
        f"Answer the question using only the context below.\n\nContext:\n{context}\n\nQuestion: {query}"
    )
    return answer.content

# 6. Setup Memory (checkpointer, replaces ConversationBufferMemory)
checkpointer = InMemorySaver()
thread_config = {"configurable": {"thread_id": "rag-demo-1"}}

# 7.Initialize Agent with Tool + Memory (replaces initialize_agent)
agent = create_agent(
    model=llm,
    tools=[LangChainRetriever],
    checkpointer=checkpointer,
)

# 8. Ask Questions (RAG-Style)
print("First Question")
res1 = agent.invoke({"messages": [{"role": "user", "content": "What is LangChain?"}]}, thread_config)
print("Answer:", res1["messages"][-1].content)

print("\nFollow-up")
res2 = agent.invoke({"messages": [{"role": "user", "content": "Who created it?"}]}, thread_config)
print("Answer:", res2["messages"][-1].content)

print("\nCombined Reasoning")
res3 = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain LangChain's use in AI workflows."}]}, thread_config
)
print("Answer:", res3["messages"][-1].content)

First Question
Answer: LangChain is a framework for building applications with large language models (LLMs). It provides tools and components to help developers create applications that leverage the capabilities of LLMs effectively. If you want to know more details or specific features of LangChain, feel free to ask!

Follow-up
Answer: LangChain was created by Harrison Chase. If you have any more questions about LangChain or its creator, feel free to ask!

Combined Reasoning
Answer: LangChain is used in AI workflows by providing a framework that supports features like Retrieval-Augmented Generation (RAG), agents, memory, and tools. These features enable developers to build sophisticated applications with large language models (LLMs), such as chatbots and document question-answering systems. By integrating these components, LangChain helps streamline the development of complex AI workflows that require interaction with external data sources, maintaining context, and performing multi-ste